In [1]:

try:
    import torch, torchvision, tqdm
    print("PyTorch/torchvision already installed.")
except Exception:
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    %pip install -q tqdm


PyTorch/torchvision already installed.


In [2]:

#@title 🔧 Imports, seeds, device
import os, random, time
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import torchvision
from torchvision import transforms

from tqdm import tqdm

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
#@title ⚙️ Configuration
class CFG:
    # Data
    source_name = "AID"
    target_name = "CLRS"
    num_classes = 7

    # Source training
    src_epochs = 20         # more epochs now that lr is gentler on the pretrained backbone
    src_batch_size = 128
    src_lr = 1e-3            # lower lr for fine-tuning ImageNet-pretrained weights on only ~227 images

    # Source-free adaptation
    sf_epochs = 30           # upper bound — sf_patience below usually stops the run well before this
    sf_batch_size = 128
    sf_lr =  2e-4               # lowered from 2e-4: was drifting/collapsing the feature extractor within ~2 epochs
    sf_patience = 4           # stop early if target accuracy hasn't improved for this many epochs —
                              # accuracy peaked at epoch 5 and only decayed afterward in the last run

    # Loss weights
    lambda_im = 0.3          # lowered from 1.0: entropy-minimization pressure was compounding with
                              # self-generated pseudo-labels into overconfident, wrong predictions
    lambda_cons = 1.0        # weak/strong consistency
    lambda_pl = 1.0          # pseudo-label CE (class-balanced)
    lambda_proto = 0.5       # prototype attraction

    # Prototype-guided class-adaptive confidence threshold
    conf_thresh_init = 0.90
    conf_thresh_min = 0.70
    conf_thresh_max = 0.95
    conf_thresh_momentum = 0.90
    min_thresh_samples = 3   # lowered from 5: with batch_size=16 and 6 classes, some classes (e.g.
                              # Industry) rarely hit 5-per-batch early on, leaving their threshold
                              # frozen at the strict initial value while other classes adapt sooner

    # Joint reliability:
    # R_c = lambda_r * confidence_reliability
    #       + (1-lambda_r) * prototype_reliability
    proto_reliability_weight = 0.50  # lambda_r
    threshold_gamma = 0.10   # halved from 0.20: keeps thresholds stricter for longer instead of
                              # relaxing acceptance to ~85-90% within the first couple epochs

    # Prototype reliability warm-up
    proto_min_count = 100    # raised from 20: prototypes were being trusted after ~1-2 batches,
                              # while still noisy — wait for more samples before trusting them

    # Consistency
    temp_cons = 0.5

    # Memory prototypes
    proto_momentum = 0.9
    feat_dim = 128

    # Misc
    num_workers = 4
    out_dir = "./cab_sfda_ckpts_A2C_V3"

os.makedirs(CFG.out_dir, exist_ok=True)
print("Config OK")

Config OK


In [ ]:
#@title 📦 Datasets and loaders  + augmentations

IMG_SIZE = 128
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

src_train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
src_test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
# Weak augmentation for target
tgt_weak_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
# Strong augmentation for target
tgt_strong_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=10, translate=(0.05,0.05), scale=(0.9,1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# 

DATA_ROOT = os.environ.get("DATA_ROOT", "D:/Downloads/data")
source_root = f"{DATA_ROOT}/data/source_aid_split"
target_root = f"{DATA_ROOT}/data/target_clrs_split"

# Load datasets (ImageFolder infers labels from the 6 class subfolders)
src_train = torchvision.datasets.ImageFolder(root=f"{source_root}/train", transform=src_train_tf)
src_test  = torchvision.datasets.ImageFolder(root=f"{source_root}/test",  transform=src_test_tf)
tgt_train_raw = torchvision.datasets.ImageFolder(root=f"{target_root}/train", transform=None)
tgt_test  = torchvision.datasets.ImageFolder(root=f"{target_root}/test", transform=transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
]))

assert src_train.classes == tgt_train_raw.classes, "Source/target class sets must match for domain adaptation"

print(f"Source (RSSCN7)     train={len(src_train)}, test={len(src_test)}")
print(f"Target (AID) train={len(tgt_train_raw)}, test={len(tgt_test)}")
print(f"Classes: {src_train.classes}")

from torch.utils.data import Dataset


class TwoViewTarget(Dataset):
    """Returns (weak_view, strong_view) and a dummy label (ignored in adaptation).

    Defined in a real module (not inline in the notebook) so DataLoader worker
    processes can pickle/import it - required for num_workers > 0 on Windows,
    where workers are spawned fresh and can't import notebook-local classes.
    """
    def __init__(self, base_ds, weak_tf, strong_tf):
        self.base = base_ds
        self.weak_tf = weak_tf
        self.strong_tf = strong_tf

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _ = self.base[idx]  # ImageFolder returns (PIL, label), but we ignore label for adaptation
        xw = self.weak_tf(img)
        xs = self.strong_tf(img)
        return xw, xs, -1  # dummy label


tgt_train = TwoViewTarget(tgt_train_raw, tgt_weak_tf, tgt_strong_tf)

# DataLoaders
src_train_loader = DataLoader(src_train, batch_size=CFG.src_batch_size, shuffle=True,  num_workers=CFG.num_workers, pin_memory=True)
src_test_loader  = DataLoader(src_test,  batch_size=CFG.src_batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
tgt_unl_loader   = DataLoader(tgt_train, batch_size=CFG.sf_batch_size, shuffle=True,  num_workers=CFG.num_workers, pin_memory=True)
tgt_test_loader  = DataLoader(tgt_test,  batch_size=CFG.sf_batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

Source (RSSCN7)     train=1750, test=750
Target (AID) train=2940, test=1260
Classes: ['Farmland', 'Forest', 'Industrial', 'Meadow', 'Parking', 'Residential', 'River']


In [ ]:
# #@title 🧠 Model (ResNet50 backbone, returns features + logits)

from torchvision import models

class ResNetBackbone(nn.Module):
    """
    ResNet-50 backbone with a custom feature projection head
    for the CAB-SFDA framework.
    """
    def __init__(self, num_classes=7, feat_dim=128, in_channels=3, use_pretrained=True):
        super().__init__()
        
        # Load pre-trained ResNet-50
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if use_pretrained else None)
        
        # 1. Feature Extractor (Backbone)
        self.features = nn.Sequential(
            self.backbone.conv1,
            self.backbone.bn1,
            self.backbone.relu,
            self.backbone.maxpool,
            self.backbone.layer1,
            self.backbone.layer2,
            self.backbone.layer3,
            self.backbone.layer4,
            self.backbone.avgpool
        )
        
        # Handle input channels
        if in_channels != 3:
            self.backbone.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # ResNet-50 outputs 2048 features after avgpool
        self.proj_dim = self.backbone.fc.in_features  # 2048
        
        # Projection head
        self.feature_head = nn.Sequential(
            nn.Linear(self.proj_dim, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True)
        )
        
        # Classifier head
        self.classifier = nn.Linear(feat_dim, num_classes)
        
        # Initialize custom layers
        nn.init.kaiming_normal_(self.feature_head[0].weight, mode='fan_out', nonlinearity='relu')
        nn.init.constant_(self.feature_head[0].bias, 0)
        nn.init.kaiming_normal_(self.classifier.weight, mode='fan_out', nonlinearity='relu')
        nn.init.constant_(self.classifier.bias, 0)
        
    def forward(self, x, return_feat=False):
        f_map = self.features(x)
        f_map = f_map.view(f_map.size(0), -1)
        f = self.feature_head(f_map)
        logits = self.classifier(f)
        if return_feat:
            return logits, f
        return logits
def accuracy(model, loader):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return 100.0 * correct / max(1, total)

src_model = ResNetBackbone(num_classes=CFG.num_classes, feat_dim=CFG.feat_dim).to(device)
print(src_model)

ResNetBackbone(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
  

In [ ]:
#@title 🏋️ Train source model  (supervised)
import json

src_results_log_path = os.path.join(CFG.out_dir, "source_training_results.json")
src_epoch_results = []

def train_source(model, train_loader, test_loader, epochs=CFG.src_epochs, lr=CFG.src_lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best = -1
    for ep in range(1, epochs+1):
        model.train()
        pbar = tqdm(train_loader, desc=f"Source epoch {ep}/{epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        acc_src = accuracy(model, test_loader)
        best = max(best, acc_src)
        print(f"[Source] Epoch {ep}: AID test acc = {acc_src:.2f}% (best {best:.2f}%)")

        # Persist progress after every epoch, so results survive an interruption.
        src_epoch_results.append({
            "epoch": ep,
            "aid_test_acc": float(acc_src),
            "best_aid_test_acc": float(best),
        })
        with open(src_results_log_path, "w") as f:
            json.dump(src_epoch_results, f, indent=2)

    return model

src_model = train_source(src_model, src_train_loader, src_test_loader, epochs=CFG.src_epochs, lr=CFG.src_lr)

# Save the fully-supervised source checkpoint; the CAB-SFDA cell loads this
# as the frozen-classifier starting point for source-free adaptation.
src_ckpt_path = os.path.join(CFG.out_dir, "source_aid_resnet50.pth")
torch.save(src_model.state_dict(), src_ckpt_path)
print("Saved source model to:", src_ckpt_path)

# Baseline accuracy on the target domain BEFORE any adaptation — this is the
# number the source-free adaptation results should improve on.
src_only_acc_on_eurosat = accuracy(src_model, tgt_test_loader)
print(f"Source-only accuracy on EuroSAT test: {src_only_acc_on_eurosat:.2f}%")

# Append the target-domain baseline to the same results file.
src_epoch_results.append({"source_only_acc_on_clrs": float(src_only_acc_on_eurosat)})
with open(src_results_log_path, "w") as f:
    json.dump(src_epoch_results, f, indent=2)

print("Saved source-training results to:", src_results_log_path)

In [ ]:
#@title 🔁 CAB-SFDA adaptation with prototype-guided class-adaptive tau_c
import json


def entropy(p, eps=1e-8):
    return -(p * (p + eps).log()).sum(dim=1)


def info_max_loss(logits):
    p = torch.softmax(logits, dim=1)
    ent_per = entropy(p)
    ent_mean = entropy(p.mean(dim=0, keepdim=True))
    return ent_per.mean() - ent_mean.mean()


def kl_divergence_with_temperature(logits_a, logits_b, T=0.5):
    pa = torch.log_softmax(logits_a / T, dim=1)
    qa = torch.softmax(logits_a / T, dim=1)
    kl = (
        qa
        * (pa - torch.log_softmax(logits_b / T, dim=1))
    ).sum(dim=1).mean()
    return kl


class ClassMemory:
    """Target-only class prototypes and confident pseudo-label counts."""

    def __init__(
        self,
        num_classes,
        feat_dim,
        momentum=0.9,
        eps=1e-6
    ):
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.m = momentum
        self.eps = eps

        self.prototypes = torch.zeros(
            num_classes,
            feat_dim,
            device=device
        )

        self.counts = torch.zeros(
            num_classes,
            device=device
        )

        self.initialized = torch.zeros(
            num_classes,
            dtype=torch.bool,
            device=device
        )

    @torch.no_grad()
    def update(self, feats, labels):
        """
        Update each class prototype using only accepted pseudo-labeled
        target features.
        """
        for c in range(self.num_classes):
            idx = (labels == c).nonzero(
                as_tuple=False
            ).flatten()

            if idx.numel() == 0:
                continue

            fmean = feats[idx].mean(dim=0)

            if not self.initialized[c]:
                self.prototypes[c] = fmean.detach()
                self.initialized[c] = True
            else:
                self.prototypes[c] = (
                    self.m * self.prototypes[c]
                    + (1.0 - self.m) * fmean.detach()
                )

            self.counts[c] += idx.numel()

    def get_weights(self):
        """
        Inverse-square-root class weighting based on accumulated
        accepted pseudo-label counts.
        """
        inv = 1.0 / torch.sqrt(
            self.counts + self.eps
        )

        inv = (
            inv
            / inv.sum().clamp_min(self.eps)
            * self.num_classes
        )

        return inv.detach()

    def proto_loss(self, feats, labels):
        """
        Attract accepted target features toward their class prototypes.
        Classes without initialized prototypes are masked out.
        """
        if feats.size(0) == 0:
            return torch.tensor(
                0.0,
                device=feats.device
            )

        valid = self.initialized[labels]

        if valid.sum() == 0:
            return torch.tensor(
                0.0,
                device=feats.device
            )

        valid_feats = feats[valid]
        valid_labels = labels[valid]
        protos = self.prototypes[valid_labels]

        return F.mse_loss(
            valid_feats,
            protos
        )


class PrototypeGuidedClassThreshold:
    """
    Prototype-guided class-adaptive confidence threshold.

    For predicted class c:

        q_c = mean prediction confidence

        r_c = normalized mean cosine agreement with prototype

        R_c = lambda_r * q_c
              + (1-lambda_r) * r_c

        raw_tau_c = tau_base - gamma * (R_c - 0.5)

    raw_tau_c is clipped to [tau_min, tau_max] and then
    temporally smoothed with an EMA.

    Before the prototype has accumulated proto_min_count accepted
    samples, prototype reliability is not trusted and r_c falls back
    to q_c. Therefore, the threshold is confidence-driven during
    prototype warm-up.

    No target labels are used.
    """

    def __init__(
        self,
        num_classes,
        init=0.90,
        tau_min=0.70,
        tau_max=0.95,
        momentum=0.90,
        min_samples=5,
        reliability_weight=0.50,
        gamma=0.20,
        proto_min_count=20,
    ):
        self.num_classes = num_classes
        self.tau_min = tau_min
        self.tau_max = tau_max
        self.momentum = momentum
        self.min_samples = min_samples
        self.reliability_weight = reliability_weight
        self.gamma = gamma
        self.base_tau = init
        self.proto_min_count = proto_min_count

        self.thresholds = torch.full(
            (num_classes,),
            float(init),
            dtype=torch.float32,
            device=device
        )

        # Diagnostic statistics
        self.conf_reliability = torch.zeros(
            num_classes,
            device=device
        )

        self.proto_reliability = torch.zeros(
            num_classes,
            device=device
        )

        self.joint_reliability = torch.zeros(
            num_classes,
            device=device
        )

        self.prototype_active = torch.zeros(
            num_classes,
            dtype=torch.bool,
            device=device
        )

    @torch.no_grad()
    def update(
        self,
        conf,
        labels,
        feats,
        memory
    ):
        """
        Update class-specific thresholds from unlabeled target
        predictions and current prototype memory.
        """

        for c in range(self.num_classes):

            idx = (labels == c)

            if idx.sum().item() < self.min_samples:
                continue

            # --------------------------------------------------
            # 1. Confidence reliability q_c
            # --------------------------------------------------
            conf_c = conf[idx]
            q_c = conf_c.mean().clamp(0.0, 1.0)

            self.conf_reliability[c] = q_c

            # --------------------------------------------------
            # 2. Prototype reliability r_c
            # --------------------------------------------------
            proto_ready = (
                bool(memory.initialized[c].item())
                and float(memory.counts[c].item())
                >= float(self.proto_min_count)
            )

            self.prototype_active[c] = proto_ready

            if proto_ready:

                feats_c = feats[idx]

                feats_c = F.normalize(
                    feats_c,
                    p=2,
                    dim=1
                )

                proto_c = F.normalize(
                    memory.prototypes[c].unsqueeze(0),
                    p=2,
                    dim=1
                )

                cosine_sim = (
                    feats_c * proto_c
                ).sum(dim=1)

                # Convert cosine similarity [-1, 1] to [0, 1].
                r_c = (
                    1.0 + cosine_sim.mean()
                ) / 2.0

                r_c = r_c.clamp(
                    0.0,
                    1.0
                )

            else:
                # During warm-up, rely only on classifier confidence.
                r_c = q_c

            self.proto_reliability[c] = r_c

            # --------------------------------------------------
            # 3. Joint reliability R_c
            # --------------------------------------------------
            lam = self.reliability_weight

            R_c = (
                lam * q_c
                + (1.0 - lam) * r_c
            ).clamp(0.0, 1.0)

            self.joint_reliability[c] = R_c

            # --------------------------------------------------
            # 4. Convert reliability into class threshold tau_c
            # --------------------------------------------------
            raw_tau = (
                self.base_tau
                - self.gamma * (R_c - 0.5)
            )

            raw_tau = raw_tau.clamp(
                self.tau_min,
                self.tau_max
            )

            # EMA temporal smoothing
            self.thresholds[c] = (
                self.momentum
                * self.thresholds[c]
                + (1.0 - self.momentum)
                * raw_tau
            )

    @torch.no_grad()
    def get_mask(
        self,
        conf,
        labels
    ):
        """
        Compare every sample against the threshold associated
        with its predicted class.
        """
        sample_thresholds = self.thresholds[
            labels
        ]

        return conf.ge(
            sample_thresholds
        )


# ------------------------------------------------------------
# Initialize student from source model
# ------------------------------------------------------------
student = ResNetBackbone(
    num_classes=CFG.num_classes,
    feat_dim=CFG.feat_dim
).to(device)


student.load_state_dict(
    torch.load(
        src_ckpt_path,
        map_location=device
    )
)

# Freeze classifier and adapt only the feature extractor.
for p in student.classifier.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.Adam(
    [
        p
        for p in list(student.features.parameters()) + list(student.feature_head.parameters())
        if p.requires_grad
    ],
    lr=CFG.sf_lr
)

# Mixed-precision training: big speedup on CUDA GPUs, no-op (disabled) on CPU.
# Only the ResNet50 forward pass runs in fp16 (autocast); everything else
# (prototype EMA, thresholds, custom losses) stays fp32 for numerical safety.
_amp_enabled = (device.type == "cuda")
sf_scaler = torch.cuda.amp.GradScaler(enabled=_amp_enabled)


# ------------------------------------------------------------
# Initialize class prototype memory
# ------------------------------------------------------------
memory = ClassMemory(
    num_classes=CFG.num_classes,
    feat_dim=CFG.feat_dim,
    momentum=CFG.proto_momentum
)


# ------------------------------------------------------------
# Initialize prototype-guided adaptive thresholds
# ------------------------------------------------------------
adaptive_thresh = PrototypeGuidedClassThreshold(
    num_classes=CFG.num_classes,
    init=CFG.conf_thresh_init,
    tau_min=CFG.conf_thresh_min,
    tau_max=CFG.conf_thresh_max,
    momentum=CFG.conf_thresh_momentum,
    min_samples=CFG.min_thresh_samples,
    reliability_weight=CFG.proto_reliability_weight,
    gamma=CFG.threshold_gamma,
    proto_min_count=CFG.proto_min_count,
)


# ------------------------------------------------------------
# Histories for analysis
# ------------------------------------------------------------
threshold_history = []
confidence_history = []
prototype_reliability_history = []
joint_reliability_history = []
prototype_active_history = []
acceptance_history = []
accuracy_history = []

best_tgt = -1.0
epochs_without_improvement = 0

# Per-epoch results are dumped to this file after every epoch, so progress
# survives even if the run is interrupted before the final cell output.
results_log_path = os.path.join(CFG.out_dir, "adaptation_results.json")
epoch_results = []

# Self-training can drift/collapse in later epochs (accuracy dropping while
# the model stays confident in its own wrong pseudo-labels). The final-epoch
# weights are NOT necessarily the best ones, so the best-so-far checkpoint is
# saved separately here, every time target accuracy improves.
best_ckpt_path = os.path.join(
    CFG.out_dir,
    "student_clrs_CAB_SFDA_best.pth"
)


# ------------------------------------------------------------
# Source-free adaptation
# ------------------------------------------------------------
for ep in range(
    1,
    CFG.sf_epochs + 1
):

    student.train()

    pbar = tqdm(
        tgt_unl_loader,
        desc=(
            f"CAB-SFDA PGAT "
            f"epoch {ep}/{CFG.sf_epochs}"
        )
    )

    accepted = 0
    observed = 0

    for xw, xs, _ in pbar:

        xw = xw.to(device)
        xs = xs.to(device)

        # ------------------------------------------------------
        # Forward weak and strong target views (fp16 under autocast for
        # speed; cast back to fp32 immediately for everything downstream)
        # ------------------------------------------------------
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=_amp_enabled):
            logits_w, feats_w = student(
                xw,
                return_feat=True
            )

            logits_s, feats_s = student(
                xs,
                return_feat=True
            )

        logits_w = logits_w.float()
        feats_w = feats_w.float()
        logits_s = logits_s.float()
        feats_s = feats_s.float()

        # ------------------------------------------------------
        # 1. Information maximization
        # ------------------------------------------------------
        loss_im = info_max_loss(
            logits_w
        )

        # ------------------------------------------------------
        # 2. Symmetric weak/strong consistency
        # ------------------------------------------------------
        kl_ws = kl_divergence_with_temperature(
            logits_w,
            logits_s,
            T=CFG.temp_cons
        )

        kl_sw = kl_divergence_with_temperature(
            logits_s,
            logits_w,
            T=CFG.temp_cons
        )

        loss_cons = 0.5 * (
            kl_ws + kl_sw
        )

        # ------------------------------------------------------
        # Target pseudo-label predictions from weak view
        # ------------------------------------------------------
        p_w = torch.softmax(
            logits_w,
            dim=1
        )

        conf, y_hat = p_w.max(
            dim=1
        )

        # ------------------------------------------------------
        # IMPORTANT:
        # Select this mini-batch using thresholds accumulated
        # from previous mini-batches.
        #
        # The current batch does NOT determine its own
        # acceptance criterion.
        # ------------------------------------------------------
        mask = adaptive_thresh.get_mask(
            conf.detach(),
            y_hat.detach()
        )

        accepted += int(
            mask.sum().item()
        )

        observed += int(
            mask.numel()
        )

        # ------------------------------------------------------
        # 3. Class-balanced pseudo-label CE on strong view
        # ------------------------------------------------------
        if mask.sum() > 0:

            class_weights = memory.get_weights()

            per_sample_ce = F.cross_entropy(
                logits_s[mask],
                y_hat[mask],
                reduction="none"
            )

            w = class_weights[
                y_hat[mask]
            ]

            loss_pl = (
                per_sample_ce * w
            ).mean()

        else:

            loss_pl = torch.tensor(
                0.0,
                device=device
            )

        # ------------------------------------------------------
        # 4. Prototype attraction
        #
        # Only prototypes that existed before the current
        # memory update are used for this batch's loss.
        # ------------------------------------------------------
        if (
            mask.sum() > 0
            and memory.initialized.any()
        ):

            loss_proto = memory.proto_loss(
                feats_w[mask],
                y_hat[mask]
            )

        else:

            loss_proto = torch.tensor(
                0.0,
                device=device
            )

        # ------------------------------------------------------
        # Total adaptation objective
        # ------------------------------------------------------
        loss = (
            CFG.lambda_im * loss_im
            + CFG.lambda_cons * loss_cons
            + CFG.lambda_pl * loss_pl
            + CFG.lambda_proto * loss_proto
        )

        sf_scaler.scale(loss).backward()
        sf_scaler.step(optimizer)
        sf_scaler.update()

        # ------------------------------------------------------
        # Update prototype memory AFTER optimization.
        #
        # Only accepted pseudo-labels enter the memory.
        # ------------------------------------------------------
        if mask.sum() > 0:

            memory.update(
                feats_w.detach()[mask],
                y_hat.detach()[mask]
            )

        # ------------------------------------------------------
        # Update prototype-guided thresholds AFTER:
        #   1. current-batch selection,
        #   2. optimization,
        #   3. prototype-memory update.
        #
        # Therefore, these updated thresholds affect only
        # subsequent mini-batches.
        # ------------------------------------------------------
        adaptive_thresh.update(
            conf.detach(),
            y_hat.detach(),
            feats_w.detach(),
            memory
        )

        pbar.set_postfix(
            IM=f"{loss_im.item():.3f}",
            Cons=f"{loss_cons.item():.3f}",
            PL=f"{loss_pl.item():.3f}",
            Proto=f"{loss_proto.item():.3f}",
            Accept=(
                f"{100.0 * accepted / max(observed, 1):.1f}%"
            )
        )

    # ----------------------------------------------------------
    # End-of-epoch diagnostics
    # ----------------------------------------------------------
    tau_epoch = (
        adaptive_thresh.thresholds
        .detach()
        .cpu()
        .numpy()
        .copy()
    )

    conf_epoch = (
        adaptive_thresh.conf_reliability
        .detach()
        .cpu()
        .numpy()
        .copy()
    )

    proto_epoch = (
        adaptive_thresh.proto_reliability
        .detach()
        .cpu()
        .numpy()
        .copy()
    )

    joint_epoch = (
        adaptive_thresh.joint_reliability
        .detach()
        .cpu()
        .numpy()
        .copy()
    )

    proto_active_epoch = (
        adaptive_thresh.prototype_active
        .detach()
        .cpu()
        .numpy()
        .copy()
    )

    acceptance_rate = (
        accepted
        / max(observed, 1)
    )

    threshold_history.append(
        tau_epoch
    )

    confidence_history.append(
        conf_epoch
    )

    prototype_reliability_history.append(
        proto_epoch
    )

    joint_reliability_history.append(
        joint_epoch
    )

    prototype_active_history.append(
        proto_active_epoch
    )

    acceptance_history.append(
        acceptance_rate
    )

    # Evaluation is for experiment reporting only.
    # Target labels are not used by the adaptation algorithm.
    acc_tgt = accuracy(
        student,
        tgt_test_loader
    )

    accuracy_history.append(
        acc_tgt
    )

    # Save the best-so-far checkpoint BEFORE overwriting best_tgt, so we
    # capture exactly the weights that produced this accuracy. Also tracks
    # epochs since the last improvement, for early stopping below.
    is_new_best = acc_tgt >= best_tgt
    best_tgt = max(
        best_tgt,
        acc_tgt
    )
    if is_new_best:
        torch.save(student.state_dict(), best_ckpt_path)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"[CAB-SFDA PGAT] Epoch {ep}: "
        f"CLRS test acc = {acc_tgt:.2f}% "
        f"(best {best_tgt:.2f}%)"
        + (" [new best, checkpoint saved]" if is_new_best else
           f" [no improvement for {epochs_without_improvement} epoch(s)]")
    )

    print(
        "Class thresholds:",
        [
            round(float(x), 3)
            for x in tau_epoch
        ]
    )

    print(
        "Confidence reliability:",
        [
            round(float(x), 3)
            for x in conf_epoch
        ]
    )

    print(
        "Prototype reliability:",
        [
            round(float(x), 3)
            for x in proto_epoch
        ]
    )

    print(
        "Joint reliability:",
        [
            round(float(x), 3)
            for x in joint_epoch
        ]
    )

    print(
        "Prototype active:",
        [
            bool(x)
            for x in proto_active_epoch
        ]
    )

    print(
        "Prototype counts:",
        [
            int(x)
            for x in (
                memory.counts
                .detach()
                .cpu()
                .tolist()
            )
        ]
    )

    print(
        f"Pseudo-label acceptance rate: "
        f"{100.0 * acceptance_rate:.2f}%"
    )

    # ----------------------------------------------------------
    # Persist this epoch's results to disk (overwrite each epoch
    # so the file always holds progress up to the latest epoch).
    # ----------------------------------------------------------
    epoch_results.append({
        "epoch": ep,
        "target_test_acc": float(acc_tgt),
        "best_target_test_acc": float(best_tgt),
        "class_thresholds": [round(float(x), 4) for x in tau_epoch],
        "confidence_reliability": [round(float(x), 4) for x in conf_epoch],
        "prototype_reliability": [round(float(x), 4) for x in proto_epoch],
        "joint_reliability": [round(float(x), 4) for x in joint_epoch],
        "prototype_active": [bool(x) for x in proto_active_epoch],
        "prototype_counts": [int(x) for x in memory.counts.detach().cpu().tolist()],
        "pseudo_label_acceptance_rate": round(100.0 * acceptance_rate, 2),
    })

    with open(results_log_path, "w") as f:
        json.dump(epoch_results, f, indent=2)

    # ----------------------------------------------------------
    # Early stopping: self-training tends to peak early and then
    # gradually drift/decay, so stop once it's clearly past its peak
    # instead of continuing to train on increasingly stale pseudo-labels.
    # ----------------------------------------------------------
    if epochs_without_improvement >= CFG.sf_patience:
        print(
            f"No improvement for {CFG.sf_patience} consecutive epochs — "
            f"stopping early after epoch {ep} (best so far: {best_tgt:.2f}%)."
        )
        break


# ------------------------------------------------------------
# Save final-epoch model (may not be the best — see best_ckpt_path)
# ------------------------------------------------------------
tgt_ckpt_path = os.path.join(
    CFG.out_dir,
    "student_clrs_CAB_SFDA_prototype_guided_adaptive_tau.pth"
)

torch.save(
    student.state_dict(),
    tgt_ckpt_path
)

print(
    "Saved final-epoch model to:",
    tgt_ckpt_path
)

print(
    f"Saved BEST model (target acc = {best_tgt:.2f}%) to:",
    best_ckpt_path
)

print(
    "Saved per-epoch results to:",
    results_log_path
)


In [ ]:
# @title 📊 Performance assessment on target test set (overall + per-class + confusion matrix + imbalance metrics)
# Results are printed to console, saved as JSON, CSV, PDF, and include a Recall/F1 bar chart + threshold sweep.

import torch, numpy as np
import matplotlib.pyplot as plt
import os, json
from datetime import datetime

def evaluate_with_cm(model, loader, num_classes=CFG.num_classes):
    """Compute overall accuracy, per-class metrics, and a confusion matrix."""
    model.eval()
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device); y = y.to(device)
            logits = model(x)
            pred = logits.argmax(1)
            for t, p in zip(y.view(-1), pred.view(-1)):
                cm[int(t.item()), int(p.item())] += 1

    # Per-class recall = diagonal / row sum
    true_counts = cm.sum(axis=1)
    per_class_recall = []
    for c in range(num_classes):
        total_c = true_counts[c]
        rec_c = (cm[c, c] / total_c) if total_c > 0 else 0.0
        per_class_recall.append(rec_c)

    # Per-class precision = diagonal / column sum
    pred_counts = cm.sum(axis=0)
    per_class_precision = []
    for c in range(num_classes):
        pred_c = pred_counts[c]
        prec_c = (cm[c, c] / pred_c) if pred_c > 0 else 0.0
        per_class_precision.append(prec_c)

    # Per-class F1
    per_class_f1 = []
    for c in range(num_classes):
        p = per_class_precision[c]
        r = per_class_recall[c]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        per_class_f1.append(f1)

    overall_acc = cm.trace() / max(1, cm.sum())
    macro_recall = np.mean(per_class_recall)
    balanced_acc = macro_recall   # macro recall = balanced accuracy
    macro_f1 = np.mean(per_class_f1)

    return {
        'cm': cm.tolist(),
        'overall_acc': float(overall_acc),
        'true_counts': true_counts.tolist(),
        'pred_counts': pred_counts.tolist(),
        'per_class_recall': [float(x) for x in per_class_recall],
        'per_class_precision': [float(x) for x in per_class_precision],
        'per_class_f1': [float(x) for x in per_class_f1],
        'macro_recall': float(macro_recall),
        'balanced_acc': float(balanced_acc),
        'macro_f1': float(macro_f1),
    }

def try_load_model(path, model_cls, **kwargs):
    if os.path.exists(path):
        m = model_cls(**kwargs).to(device)
        m.load_state_dict(torch.load(path, map_location=device))
        return m
    return None

def pct(x): return f"{100.0*x:.2f}%"

# ---------- Main evaluation ----------
results = {}

# Load models
try:
    _ = student
    res = evaluate_with_cm(student, tgt_test_loader)
    results["Adapted (in-memory student)"] = res
except NameError:
    pass

adapt_ckpt = os.path.join(CFG.out_dir, "student_clrs_CAB_SFDA.pth")
m_adapt = try_load_model(adapt_ckpt, ResNetBackbone, num_classes=CFG.num_classes, feat_dim=CFG.feat_dim)
if m_adapt is not None and "AID → CLRS" not in results:
    res = evaluate_with_cm(m_adapt, tgt_test_loader)
    results["Adapted (ckpt)"] = res

source_ckpt = os.path.join(CFG.out_dir, "source_aid_resnet50.pth")
m_src = try_load_model(source_ckpt, ResNetBackbone, num_classes=CFG.num_classes, feat_dim=CFG.feat_dim)
if m_src is not None:
    res = evaluate_with_cm(m_src, tgt_test_loader)
    results["Source-only on target"] = res

# ---------- Print report to console ----------
print("\n=== Target Performance Report (CLRS test) ===\n")
for name, res in results.items():
    print(f"[{name}]")
    print(f"Overall accuracy: {pct(res['overall_acc'])}")
    print(f"Balanced accuracy (macro recall): {pct(res['balanced_acc'])}")
    print(f"Macro F1: {pct(res['macro_f1'])}")
    print("Per-class recall:", ", ".join(f"{c}:{pct(r)}" for c, r in enumerate(res['per_class_recall'])))
    print("Per-class precision:", ", ".join(f"{c}:{pct(p)}" for c, p in enumerate(res['per_class_precision'])))
    print("Per-class F1:", ", ".join(f"{c}:{pct(f)}" for c, f in enumerate(res['per_class_f1'])))
    print()

# ---------- Save all results to JSON ----------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = os.path.join(CFG.out_dir, "evaluation_results.json")

full_report = {
    "timestamp": timestamp,
    "domain_shift": f"{CFG.source_name} → {CFG.target_name}",
    "num_classes": CFG.num_classes,
    "results": results
}

with open(json_path, "w") as f:
    json.dump(full_report, f, indent=2)
print(f"\nSaved full evaluation results to: {json_path}")

# ---------- Class distribution & imbalance ----------
if results:
    best_name = max(results.items(), key=lambda kv: kv[1]['overall_acc'])[0]
    best_res = results[best_name]
    cm_array = np.array(best_res['cm'])
    true_counts = best_res['true_counts']

    try:
        class_names = tgt_test_loader.dataset.classes
    except AttributeError:
        class_names = [f"Class {i}" for i in range(CFG.num_classes)]

    print("\n=== Class Distribution (Test Set) ===")
    for i, name in enumerate(class_names):
        print(f"  {name}: {true_counts[i]} samples")

    max_count = max(true_counts) if true_counts else 1
    min_count = min(true_counts) if true_counts and min(true_counts) > 0 else 1
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}")

    threshold = max_count / 2
    minority = [class_names[i] for i, cnt in enumerate(true_counts) if cnt < threshold]
    if minority:
        print(f"Minority classes (count < {threshold:.1f}): {', '.join(minority)}")
    else:
        print("No severely underrepresented classes.")
    print()

    print("F1 scores for each class:")
    for i, name in enumerate(class_names):
        f1 = best_res['per_class_f1'][i] * 100
        print(f"  {name}: {f1:.2f}%")
    print()

    # ---------- Save confusion matrix as CSV ----------
    csv_path = os.path.join(CFG.out_dir, "confusion_matrix.csv")
    np.savetxt(csv_path, cm_array, delimiter=",", fmt="%d")
    print(f"Saved confusion matrix (CSV) to: {csv_path}")

    # ---------- Plot confusion matrix ----------
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm_array, interpolation='nearest', cmap='Blues')
    ax.set_title("Confusion Matrix", fontweight='bold')
    ax.set_xlabel("Predicted", fontweight='bold')
    ax.set_ylabel("True", fontweight='bold')

    num_classes = CFG.num_classes
    ax.set_xticks(range(num_classes))
    ax.set_xticklabels([str(i) for i in range(num_classes)], fontweight='bold')
    ax.set_yticks(range(num_classes))
    ax.set_yticklabels([str(i) for i in range(num_classes)], fontweight='bold')

    thresh = cm_array.max() / 2.0 if cm_array.max() > 0 else 0
    for i in range(cm_array.shape[0]):
        for j in range(cm_array.shape[1]):
            value = cm_array[i, j]
            ax.text(j, i, f"{value:d}",
                    ha="center", va="center",
                    color="white" if value > thresh else "black",
                    fontsize=9, fontweight='bold')
    fig.colorbar(im, ax=ax)
    plt.tight_layout()

    cm_fig_path = os.path.join(CFG.out_dir, "confusion_matrix.pdf")
    plt.savefig(cm_fig_path, bbox_inches="tight", dpi=300)
    print("Saved annotated confusion matrix (PDF) to:", cm_fig_path)
    plt.show()

    # ========== Recall/F1 bar chart ==========
    recalls = best_res['per_class_recall']
    f1s = best_res['per_class_f1']

    x_labels = class_names if class_names else [str(i) for i in range(CFG.num_classes)]

    fig2, ax2 = plt.subplots(figsize=(12, 6))
    x = np.arange(len(x_labels))
    width = 0.35

    bars1 = ax2.bar(x - width/2, recalls, width, label='Recall',
                    color='#1f77b4', edgecolor='black', linewidth=0.8)
    bars2 = ax2.bar(x + width/2, f1s, width, label='F1 Score',
                    color='#ff7f0e', edgecolor='black', linewidth=0.8)

    for bar in bars1:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height/2,
                 f'{height:.2f}', ha='center', va='center',
                 fontsize=10, fontweight='bold', color='white')
    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height/2,
                 f'{height:.2f}', ha='center', va='center',
                 fontsize=10, fontweight='bold', color='white')

    ax2.axhline(y=0.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='0.5 baseline')

    ax2.set_xlabel('Class', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax2.set_title('Per-class Recall and F1 Scores', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)
    ax2.set_ylim(0, 1.1)
    ax2.legend(loc='upper left', frameon=True, fontsize=10)
    ax2.grid(axis='y', linestyle='--', alpha=0.6)

    plt.tight_layout()

    recall_f1_fig_path = os.path.join(CFG.out_dir, "recall_f1_scores.pdf")
    plt.savefig(recall_f1_fig_path, bbox_inches="tight", dpi=300)
    print("Saved improved Recall/F1 bar chart (PDF) to:", recall_f1_fig_path)
    plt.show()

    # ========== Threshold sweep (no pandas) ==========
    print("\n=== Threshold Sweep (Best Adapted Model) ===\n")

    # Load the best adapted model
    best_ckpt_path = os.path.join(CFG.out_dir, "student_clrs_CAB_SFDA_best.pth")
    if os.path.exists(best_ckpt_path):
        sweep_model = ResNetBackbone(num_classes=CFG.num_classes, feat_dim=CFG.feat_dim).to(device)
        sweep_model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
        sweep_model.eval()
    else:
        try:
            sweep_model = student
        except NameError:
            print("No model found for threshold sweep.")
            sweep_model = None

    if sweep_model is not None:
        thresholds = np.arange(0.1, 1.0, 0.1)
        accs = []
        acceptance_rates = []

        with torch.no_grad():
            for thresh in thresholds:
                correct = 0
                total = 0
                passed = 0
                for x, y in tgt_test_loader:
                    x, y = x.to(device), y.to(device)
                    logits = sweep_model(x)
                    probs = torch.softmax(logits, dim=1)
                    conf, pred = probs.max(dim=1)
                    mask = conf >= thresh
                    if mask.sum() == 0:
                        continue
                    passed += mask.sum().item()
                    correct += (pred[mask] == y[mask]).sum().item()
                    total += y[mask].numel()
                acc = correct / max(1, total) * 100
                acc_rate = passed / max(1, total) * 100
                accs.append(acc)
                acceptance_rates.append(acc_rate)

        # Print table using plain formatting (no pandas)
        print("Threshold   Accuracy (%)   Acceptance Rate (%)")
        print("---------   ------------   ------------------")
        for th, acc, ar in zip(thresholds, accs, acceptance_rates):
            print(f"   {th:.1f}         {acc:6.2f}             {ar:6.2f}")
        print()

        # Save CSV (using np.savetxt with header)
        sweep_data = np.column_stack((thresholds, accs, acceptance_rates))
        sweep_csv_path = os.path.join(CFG.out_dir, "threshold_sweep_results.csv")
        header = "Threshold, Accuracy (%), Acceptance Rate (%)"
        np.savetxt(sweep_csv_path, sweep_data, delimiter=",", fmt="%.2f", header=header, comments='')
        print(f"Saved threshold sweep results to: {sweep_csv_path}")

        # Plot sweep
        fig3, ax3 = plt.subplots(figsize=(8, 5))
        ax3.set_xlabel('Confidence Threshold', fontsize=12)
        ax3.set_ylabel('Accuracy (%)', color='tab:blue', fontsize=12)
        ax3.plot(thresholds, accs, marker='o', color='tab:blue', label='Accuracy')
        ax3.tick_params(axis='y', labelcolor='tab:blue')
        ax3.grid(axis='x', linestyle='--', alpha=0.5)

        ax4 = ax3.twinx()
        ax4.set_ylabel('Acceptance Rate (%)', color='tab:red', fontsize=12)
        ax4.plot(thresholds, acceptance_rates, marker='s', color='tab:red', label='Acceptance Rate')
        ax4.tick_params(axis='y', labelcolor='tab:red')

        fig3.suptitle('Accuracy vs. Confidence Threshold (Best Adapted Model)', fontweight='bold')
        fig3.tight_layout()
        sweep_plot_path = os.path.join(CFG.out_dir, "threshold_sweep.pdf")
        plt.savefig(sweep_plot_path, dpi=300, bbox_inches="tight")
        print("Saved threshold sweep plot to:", sweep_plot_path)
        plt.show()

else:
    print("No models found to evaluate.")

In [ ]:
# @title 📊 Performance assessment on target test set (overall + per-class + confusion matrix + imbalance metrics)
# Results are printed to console, saved as JSON, CSV, PDF, and include a Recall/F1 bar chart.

import torch, numpy as np
import matplotlib.pyplot as plt
import os, json
from datetime import datetime

def evaluate_with_cm(model, loader, num_classes=CFG.num_classes):
    """Compute overall accuracy, per-class metrics, and a confusion matrix."""
    model.eval()
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device); y = y.to(device)
            logits = model(x)
            pred = logits.argmax(1)
            for t, p in zip(y.view(-1), pred.view(-1)):
                cm[int(t.item()), int(p.item())] += 1

    # Per-class recall = diagonal / row sum
    true_counts = cm.sum(axis=1)
    per_class_recall = []
    for c in range(num_classes):
        total_c = true_counts[c]
        rec_c = (cm[c, c] / total_c) if total_c > 0 else 0.0
        per_class_recall.append(rec_c)

    # Per-class precision = diagonal / column sum
    pred_counts = cm.sum(axis=0)
    per_class_precision = []
    for c in range(num_classes):
        pred_c = pred_counts[c]
        prec_c = (cm[c, c] / pred_c) if pred_c > 0 else 0.0
        per_class_precision.append(prec_c)

    # Per-class F1
    per_class_f1 = []
    for c in range(num_classes):
        p = per_class_precision[c]
        r = per_class_recall[c]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        per_class_f1.append(f1)

    overall_acc = cm.trace() / max(1, cm.sum())
    macro_recall = np.mean(per_class_recall)
    balanced_acc = macro_recall   # macro recall = balanced accuracy
    macro_f1 = np.mean(per_class_f1)

    return {
        'cm': cm.tolist(),
        'overall_acc': float(overall_acc),
        'true_counts': true_counts.tolist(),
        'pred_counts': pred_counts.tolist(),
        'per_class_recall': [float(x) for x in per_class_recall],
        'per_class_precision': [float(x) for x in per_class_precision],
        'per_class_f1': [float(x) for x in per_class_f1],
        'macro_recall': float(macro_recall),
        'balanced_acc': float(balanced_acc),
        'macro_f1': float(macro_f1),
    }

def try_load_model(path, model_cls, **kwargs):
    if os.path.exists(path):
        m = model_cls(**kwargs).to(device)
        m.load_state_dict(torch.load(path, map_location=device))
        return m
    return None

def pct(x): return f"{100.0*x:.2f}%"

# ---------- Main evaluation ----------
results = {}

# Load models
try:
    _ = student
    res = evaluate_with_cm(student, tgt_test_loader)
    results["Adapted (in-memory student)"] = res
except NameError:
    pass

adapt_ckpt = os.path.join(CFG.out_dir, "student_clrs_CAB_SFDA.pth")
m_adapt = try_load_model(adapt_ckpt, ResNetBackbone, num_classes=CFG.num_classes, feat_dim=CFG.feat_dim)
if m_adapt is not None and "AID → CLRS" not in results:
    res = evaluate_with_cm(m_adapt, tgt_test_loader)
    results["Adapted (ckpt)"] = res

source_ckpt = os.path.join(CFG.out_dir, "source_aid_resnet50.pth")
m_src = try_load_model(source_ckpt, ResNetBackbone, num_classes=CFG.num_classes, feat_dim=CFG.feat_dim)
if m_src is not None:
    res = evaluate_with_cm(m_src, tgt_test_loader)
    results["Source-only on target"] = res

# ---------- Print report to console ----------
print("\n=== Target Performance Report (CLRS test) ===\n")
for name, res in results.items():
    print(f"[{name}]")
    print(f"Overall accuracy: {pct(res['overall_acc'])}")
    print(f"Balanced accuracy (macro recall): {pct(res['balanced_acc'])}")
    print(f"Macro F1: {pct(res['macro_f1'])}")
    print("Per-class recall:", ", ".join(f"{c}:{pct(r)}" for c, r in enumerate(res['per_class_recall'])))
    print("Per-class precision:", ", ".join(f"{c}:{pct(p)}" for c, p in enumerate(res['per_class_precision'])))
    print("Per-class F1:", ", ".join(f"{c}:{pct(f)}" for c, f in enumerate(res['per_class_f1'])))
    print()

# ---------- Save all results to JSON ----------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = os.path.join(CFG.out_dir, f"evaluation_results_{timestamp}.json")

full_report = {
    "timestamp": timestamp,
    "domain_shift": f"{CFG.source_name} → {CFG.target_name}",
    "num_classes": CFG.num_classes,
    "results": results
}

with open(json_path, "w") as f:
    json.dump(full_report, f, indent=2)
print(f"\nSaved full evaluation results to: {json_path}")

# ---------- Class distribution & imbalance ----------
if results:
    best_name = max(results.items(), key=lambda kv: kv[1]['overall_acc'])[0]
    best_res = results[best_name]
    cm_array = np.array(best_res['cm'])
    true_counts = best_res['true_counts']

    try:
        class_names = tgt_test_loader.dataset.classes
    except AttributeError:
        class_names = [f"Class {i}" for i in range(CFG.num_classes)]

    print("\n=== Class Distribution (Test Set) ===")
    for i, name in enumerate(class_names):
        print(f"  {name}: {true_counts[i]} samples")

    max_count = max(true_counts) if true_counts else 1
    min_count = min(true_counts) if true_counts and min(true_counts) > 0 else 1
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}")

    threshold = max_count / 2
    minority = [class_names[i] for i, cnt in enumerate(true_counts) if cnt < threshold]
    if minority:
        print(f"Minority classes (count < {threshold:.1f}): {', '.join(minority)}")
    else:
        print("No severely underrepresented classes.")
    print()

    print("F1 scores for each class:")
    for i, name in enumerate(class_names):
        f1 = best_res['per_class_f1'][i] * 100
        print(f"  {name}: {f1:.2f}%")
    print()

    # ---------- Save confusion matrix as CSV ----------
    csv_path = os.path.join(CFG.out_dir, f"confusion_matrix.csv")
    np.savetxt(csv_path, cm_array, delimiter=",", fmt="%d")
    print(f"Saved confusion matrix (CSV) to: {csv_path}")

    # ---------- Plot confusion matrix with bold annotations ----------
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm_array, interpolation='nearest', cmap='Blues')
    ax.set_title(f"Confusion Matrix (AID → CLRS)", fontweight='bold')
    ax.set_xlabel("Predicted", fontweight='bold')
    ax.set_ylabel("True", fontweight='bold')

    num_classes = CFG.num_classes
    ax.set_xticks(range(num_classes))
    ax.set_xticklabels([str(i) for i in range(num_classes)], fontweight='bold')
    ax.set_yticks(range(num_classes))
    ax.set_yticklabels([str(i) for i in range(num_classes)], fontweight='bold')

    thresh = cm_array.max() / 2.0 if cm_array.max() > 0 else 0
    for i in range(cm_array.shape[0]):
        for j in range(cm_array.shape[1]):
            value = cm_array[i, j]
            ax.text(j, i, f"{value:d}",
                    ha="center", va="center",
                    color="white" if value > thresh else "black",
                    fontsize=9, fontweight='bold')
    fig.colorbar(im, ax=ax)
    plt.tight_layout()

    cm_fig_path = os.path.join(CFG.out_dir, f"confusion_matrix.pdf")
    plt.savefig(cm_fig_path, bbox_inches="tight")
    print("Saved annotated confusion matrix (PDF) to:", cm_fig_path)
    plt.show()

    # ========== NEW: Bar chart for Per-class Recall and F1 ==========
    recalls = best_res['per_class_recall']
    f1s = best_res['per_class_f1']

    fig2, ax2 = plt.subplots(figsize=(10, 6))
    x = np.arange(len(class_names))
    width = 0.35

    bars1 = ax2.bar(x - width/2, recalls, width, label='Recall', color='skyblue', edgecolor='black', linewidth=0.5)
    bars2 = ax2.bar(x + width/2, f1s, width, label='F1 Score', color='lightcoral', edgecolor='black', linewidth=0.5)

    # Add value annotations on top of bars
    for bar in bars1:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax2.set_xlabel('Class Index', fontweight='bold')
    ax2.set_ylabel('Score', fontweight='bold')
    ax2.set_title(f'Per-class Recall and F1 Scores', fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels([str(i) for i in range(num_classes)], fontweight='bold')  # numeric labels
    ax2.set_ylim(0, 1.1)
    ax2.legend(loc='upper left', frameon=True)
    ax2.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()

    recall_f1_fig_path = os.path.join(CFG.out_dir, f"recall_f1_scores.pdf")
    plt.savefig(recall_f1_fig_path, bbox_inches="tight")
    print("Saved Recall/F1 bar chart (PDF) to:", recall_f1_fig_path)
    plt.show()

else:
    print("No models found to evaluate.")

In [ ]:
#@title 📈 Plot class-wise threshold trajectories
import os
import numpy as np
import matplotlib.pyplot as plt

tau_hist = np.asarray(threshold_history)

if tau_hist.size == 0:
    print("Run adaptation first.")
else:
    plt.figure(figsize=(10, 6))

    epochs = np.arange(
        1,
        tau_hist.shape[0] + 1
    )

    for c in range(
        tau_hist.shape[1]
    ):
        plt.plot(
            epochs,
            tau_hist[:, c],
            marker="o",
            label=f"Class {c}"
        )

    plt.xlabel("Adaptation epoch")
    plt.ylabel("Class-adaptive threshold")
    plt.title(
        "Prototype-Guided Class-Adaptive Threshold Trajectories (AID → CLRS)"
    )
    plt.grid(True, alpha=0.3)
    plt.legend(
        ncol=2,
        fontsize=9
    )
    plt.tight_layout()

    fig_path = os.path.join(CFG.out_dir, "class_threshold_trajectories.pdf")
    plt.savefig(fig_path, bbox_inches="tight")
    print("Saved figure to:", fig_path)

    plt.show()

In [ ]:
#@title 🎨 t-SNE feature-space visualization (domain & category alignment)
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import numpy as np
import os

# ---------------------- Feature Extraction ----------------------
def extract_features(model, loader):
    model.eval()
    feats_list, labels_list = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits, feats = model(x, return_feat=True)  # works with ResNetBackbone
            feats_list.append(feats.cpu())
            labels_list.append(y.cpu())
    return torch.cat(feats_list), torch.cat(labels_list)


# ---------------------- Plot Function ----------------------
def plot_embeddings(feats, labels, label_names, title, method="tsne", save_path=None):
    feats_np = feats.numpy()
    labels_np = labels.numpy()

    if method == "tsne":
        reducer = TSNE(
            n_components=2,
            perplexity=30,
            init="pca",
            random_state=42
        )
    else:
        reducer = PCA(n_components=2)

    emb = reducer.fit_transform(feats_np)

    plt.figure(figsize=(6,5))
    for lab in np.unique(labels_np):
        idx = labels_np == lab
        plt.scatter(
            emb[idx,0],
            emb[idx,1],
            label=label_names[int(lab)],
            s=10,
            alpha=0.6
        )
    plt.title(title)
    plt.legend(fontsize=8)
    plt.axis("off")

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()


# ---------------------- Label Names ----------------------
category_names = ['Farmland', 'Forest', 'Industrial', 'Meadow', 'Parking', 'Residential', 'River']
domain_names   = ["Source (AID)", "Target (CLRS)"]

assert len(category_names) == CFG.num_classes


# ---------------------- Load Source Model ----------------------
src_ckpt_viz_path = os.path.join(CFG.out_dir, "source_aid_resnet50.pth")
if not os.path.exists(src_ckpt_viz_path):
    raise FileNotFoundError(
        f"No source checkpoint at {src_ckpt_viz_path}. "
        "Run the source-training cell first."
    )

src_model_viz = ResNetBackbone(
    num_classes=CFG.num_classes,
    feat_dim=CFG.feat_dim,
    use_pretrained=False  # weights come from the checkpoint below, no need to download ImageNet init
).to(device)

src_model_viz.load_state_dict(torch.load(src_ckpt_viz_path, map_location=device))


# ---------------------- Load Adapted Model ----------------------
# Prefer the best-accuracy checkpoint; fall back to the final-epoch one if the
# adaptation cell was run before best-checkpoint saving was added, or if
# training is still in progress.
adapt_ckpt_candidates = [
    ("best", os.path.join(CFG.out_dir, "student_clrs_CAB_SFDA_best.pth")),
    ("final-epoch", os.path.join(CFG.out_dir, "student_clrs_CAB_SFDA_prototype_guided_adaptive_tau.pth")),
]

adapt_ckpt_viz_path = None
for label, path in adapt_ckpt_candidates:
    if os.path.exists(path):
        adapt_ckpt_viz_path = path
        print(f"Using {label} adapted checkpoint: {path}")
        break

if adapt_ckpt_viz_path is None:
    raise FileNotFoundError(
        "No adapted checkpoint found in "
        f"{CFG.out_dir}. Run the CAB-SFDA adaptation cell first."
    )

adapt_model_viz = ResNetBackbone(
    num_classes=CFG.num_classes,
    feat_dim=CFG.feat_dim,
    use_pretrained=False
).to(device)

adapt_model_viz.load_state_dict(torch.load(adapt_ckpt_viz_path, map_location=device))

# ---------------------- Extract Features ----------------------
src_feats, src_labels = extract_features(src_model_viz, src_test_loader)
tgt_feats_pre, tgt_labels_pre = extract_features(src_model_viz, tgt_test_loader)
tgt_feats_post, tgt_labels_post = extract_features(adapt_model_viz, tgt_test_loader)

# Domain labels
src_domain_labels = torch.zeros_like(src_labels)
tgt_domain_labels_pre  = torch.ones_like(tgt_labels_pre)
tgt_domain_labels_post = torch.ones_like(tgt_labels_post)

# ---------------------- Plot Domain-Level Distribution ----------------------
plot_embeddings(
    torch.cat([src_feats, tgt_feats_pre]),
    torch.cat([src_domain_labels, tgt_domain_labels_pre]),
    domain_names,
    title="Domain Alignment (Before Adaptation)",
    method="tsne",
    save_path=os.path.join(CFG.out_dir, "tsne_domain_before.pdf")
)

plot_embeddings(
    torch.cat([src_feats, tgt_feats_post]),
    torch.cat([src_domain_labels, tgt_domain_labels_post]),
    domain_names,
    title="Domain Alignment (After Adaptation)",
    method="tsne",
    save_path=os.path.join(CFG.out_dir, "tsne_domain_after.pdf")
)

# ---------------------- Plot Category-Level Distribution ----------------------
plot_embeddings(
    torch.cat([src_feats, tgt_feats_pre]),
    torch.cat([src_labels, tgt_labels_pre]),
    category_names,
    title="Category Alignment (AID → CLRS) - Before Adaptation ",
    method="tsne",
    save_path=os.path.join(CFG.out_dir, "tsne_category_before.pdf")
)

plot_embeddings(
    torch.cat([src_feats, tgt_feats_post]),
    torch.cat([src_labels, tgt_labels_post]),
    category_names,
    title="Category Alignment (AID → CLRS) - After Adaptation",
    method="tsne",
    save_path=os.path.join(CFG.out_dir, "tsne_category_after.pdf")
)
